In [24]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import mmap
import random
import pickle

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

batch_size = 64
block_size = 128
max_iters = 5000
learning_rate = 3e-4
eval_iters = 500
n_embd = 384
n_layer = 8
n_head = 8
dropout = 0.2 

cuda


In [25]:
chars = ""
with open("openwebtext/vocab.txt", 'r', encoding='utf-8') as f:
    text = f.read()
    chars = sorted(list(set(text)))
    
vocab_size = len(chars)

In [26]:
string_to_int= { ch:i for i,ch in enumerate(chars) }
int_to_string= { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ' '.join([int_to_string[i] for i in l])


In [27]:
def get_random_chunk(split):
   filename = "openwebtext/train_split.txt" if split == 'train' else "openwebtext/val_split.txt"
   with open(filename, 'rb') as f:
       with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
           # determine the size of the file and randomly select a starting point
           file_size = len(mm)
           start_pos = random.randint(0, file_size - block_size * batch_size)

           # seek to the random starting position and read a chunk of data
           mm.seek(start_pos)
           block = mm.read(block_size * batch_size-1)

           # decode the block and convert it to a tensor
           decoded_block = block.decode('utf-8', errors='ignore').replace('\r', ' ')

           #train and test splits
           data = torch.tensor(encode(decoded_block), dtype=torch.long)
           return data


def get_batch(split):
    data = get_random_chunk(split)
    # generate random starting indices for the batch
    ix = torch.randint(0, len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


In [28]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out
        

In [33]:
class Head(nn.Module):
    """ one head of self-attention"""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size,bias=False)
        self.value = nn.Linear(n_embd, head_size,bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step,channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x) # (B, T, hs)
        q = self.query(x) # (B, T, hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B, T, hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel"""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range (num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim = -1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4* n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation"""

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the nuumber of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x

class GPTLLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) # nn.Embedding(vocab_size, vocab_size) is basically a lookup table
        self.position_embedding_table= nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head = n_head) for _ in range (n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)
    

    def forward(self, index, targets = None):
        B, T = index.shape
        
        
        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(index) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
        x = tok_emb + pos_emb # (B, T, C)
        x = self.blocks(x) # (B, T, C)
        x= self.ln_f(x) # (B, T, C)
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):

        # Only feed the last 64 tokens
            index_cond = index[:, -block_size:]

            logits, loss = self.forward(index_cond)

            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)

            index_next = torch.multinomial(probs, num_samples=1)

            index = torch.cat((index, index_next), dim=1)

        return index

model = GPTLLM(vocab_size)
print('loading model parameters...')
with open('model-01.pkl', 'rb') as f:
    model = pickle.load(f)
print('loaded successfully')
m = model.to(device)

#context = torch.zeros((1,1), dtype=torch.long, device=device)
#generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
#print(generated_chars)

loading model parameters...
loaded successfully


In [32]:
# creating a pytorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}") #the f is for the decimal spot
        
    # sample a batch of data
    xb, yb =  get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()
print(loss.item())

with open('model-01.pkl', 'wb') as f:
    pickle.dump(model, f)

step: 0, train loss: 10.494, val loss: 10.492
step: 500, train loss: 2.165, val loss: 2.156
step: 1000, train loss: 1.782, val loss: 1.787
step: 1500, train loss: 1.612, val loss: 1.599
step: 2000, train loss: 1.502, val loss: 1.509
step: 2500, train loss: 1.435, val loss: 1.454
step: 3000, train loss: 1.408, val loss: 1.395
step: 3500, train loss: 1.377, val loss: 1.342
step: 4000, train loss: 1.336, val loss: 1.331
step: 4500, train loss: 1.346, val loss: 1.351
1.0527700185775757


In [21]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=5000)[0].tolist())
print(generated_chars)

                                                            u s t a r                                                                                                                                                 0 0 0 0 6 4 4   0 0 0 0 0 0 0   0 0 0 0 0 0 0   0 0 0 0 0 0 1 2 7 0 4   0 0 0 0 0 0 0 0 0   0 1 5 5 4     0                                                                                                                                                                                                                                                                   N e h o l d   B o w a r d   G o s i g n e r   s t o r y   f o r   a   m u c h   s i n g i n g   t h a t ,   b u t   t h e i r   B u i l t y   a l m o   d i f f i c u l t   d e s p i t e d   b u t t   t h a t   b a s e   o   t h e   I s e l l i a t i o n   r e s u l t s   w i t h   f a m i l y   h e a r t   t o   c r e a t e   D L A L   C u s   a n d   —   c o u t e s   M r   r e t a i d .   W i d e l p o w e r e d   